In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# --- 1. SETUP ---
MAIN_DATA_FILE = 'C:/Users/511232/Desktop/DSS/MERGING GOOGLESHEETS QUESTIONNAIRES/codes/arabic_questionnaires.xlsx'
CRITERIA_FILE = 'C:/Users/511232/Desktop/criterias.xlsx'

# Load data
main_df = pd.read_excel(MAIN_DATA_FILE)
criteria_df = pd.read_excel(CRITERIA_FILE).dropna(subset=['Indicator_Ar'])[['Indicator_Ar', 'criteria']]

# Force numeric values (Invalid text becomes NaN)
main_df['val_numeric'] = pd.to_numeric(main_df['القيمة'], errors='coerce')

In [ ]:
# --- 2. GENERAL AVAILABILITY ---
# Collapse to Year level (max picks the value if it exists)
gen_yearly = main_df.groupby(['المؤشر', 'الدولة', 'السنة'])['val_numeric'].max().reset_index()
gen_yearly['exist'] = gen_yearly['val_numeric'].notna().astype(int)

# Create Brackets
conds = [
    (gen_yearly['السنة'] <= 2014),
    (gen_yearly['السنة'] <= 2019),
    (gen_yearly['السنة'] <= 2026)
]
bins = ['2010-2014', '2015-2019', '2020-2026']
gen_yearly['bracket'] = np.select(conds, bins, default='Other')

# Sum valid years per bracket
gen_brk = gen_yearly[gen_yearly['bracket'] != 'Other'].groupby(['المؤشر', 'الدولة', 'bracket'])['exist'].sum().reset_index()

# Merge criteria and check if threshold met
gen_merged = pd.merge(gen_brk, criteria_df, left_on='المؤشر', right_on='Indicator_Ar', how='left')
gen_merged['is_met'] = (gen_merged['exist'] >= gen_merged['criteria']).astype(int)

# FINAL SIMPLE CHECK
# Group by Ind/Country: If you have 3 rows (brackets) and 3 'is_met' values, you are Available.
final_gen = gen_merged.groupby(['المؤشر', 'الدولة']).filter(lambda x: len(x) == 3 and x['is_met'].all())
final_gen = final_gen[['المؤشر', 'الدولة']].drop_duplicates()
final_gen['general_availability'] = 1

# Merge back to original Ind/Country list to show 0 for the failures
all_pairs = gen_merged[['المؤشر', 'الدولة']].drop_duplicates()
final_gen = pd.merge(all_pairs, final_gen, on=['المؤشر', 'الدولة'], how='left').fillna(0)
final_gen.to_excel('general_availability.xlsx', index=False)

In [ ]:
# --- 3. NATIONALITY AVAILABILITY ---
# Remove blanks
df_nat = main_df.dropna(subset=['المواطنة', 'val_numeric'])
df_nat = df_nat[~df_nat['المواطنة'].isin(['Not applicable', 'غير متاح', 'Total'])]

# Collapse to Year (Confirm if EITHER has data)
nat_yearly = df_nat.groupby(['المؤشر', 'الدولة', 'السنة'])['val_numeric'].max().reset_index()
nat_yearly['bracket'] = np.select(conds, bins, default='Other')

# Sum years in bracket
nat_brk = nat_yearly[nat_yearly['bracket'] != 'Other'].groupby(['المؤشر', 'الدولة', 'bracket']).size().reset_index(name='count')
nat_merged = pd.merge(nat_brk, criteria_df, left_on='المؤشر', right_on='Indicator_Ar', how='left')
nat_merged['is_met'] = (nat_merged['count'] >= nat_merged['criteria']).astype(int)

# Final Check
final_nat = nat_merged.groupby(['المؤشر', 'الدولة']).filter(lambda x: len(x) == 3 and x['is_met'].all())
final_nat = final_nat[['المؤشر', 'الدولة']].drop_duplicates()
final_nat['nationality_availability'] = 1

all_pairs_nat = nat_merged[['المؤشر', 'الدولة']].drop_duplicates()
final_nat = pd.merge(all_pairs_nat, final_nat, on=['المؤشر', 'الدولة'], how='left').fillna(0)
final_nat.to_excel('nationality_availability.xlsx', index=False)

In [ ]:
# --- 4. SEX AVAILABILITY ---
# Remove blanks
df_sex = main_df.dropna(subset=['الجنس', 'val_numeric'])
df_sex = df_sex[~df_sex['الجنس'].isin(['Not applicable', 'غير متاح', 'Total'])]

# Collapse to Year
sex_yearly = df_sex.groupby(['المؤشر', 'الدولة', 'السنة'])['val_numeric'].max().reset_index()
sex_yearly['bracket'] = np.select(conds, bins, default='Other')

# Sum years in bracket
sex_brk = sex_yearly[sex_yearly['bracket'] != 'Other'].groupby(['المؤشر', 'الدولة', 'bracket']).size().reset_index(name='count')
sex_merged = pd.merge(sex_brk, criteria_df, left_on='المؤشر', right_on='Indicator_Ar', how='left')
sex_merged['is_met'] = (sex_merged['count'] >= sex_merged['criteria']).astype(int)

# Final Check
final_sex = sex_merged.groupby(['المؤشر', 'الدولة']).filter(lambda x: len(x) == 3 and x['is_met'].all())
final_sex = final_sex[['المؤشر', 'الدولة']].drop_duplicates()
final_sex['sex_availability'] = 1

all_pairs_sex = sex_merged[['المؤشر', 'الدولة']].drop_duplicates()
final_sex = pd.merge(all_pairs_sex, final_sex, on=['المؤشر', 'الدولة'], how='left').fillna(0)
final_sex.to_excel('sex_availability.xlsx', index=False)

print("Process Complete. 3 files saved.")

In [ ]:
# --- 5. MERGE & SAVE ---
results = pd.merge(final_gen[['المؤشر', 'الدولة', 'general_availability']], 
                    final_nat[['المؤشر', 'الدولة', 'nationality_availability']], on=['المؤشر', 'الدولة'], how='left')
results = pd.merge(results, 
                    final_sex[['المؤشر', 'الدولة', 'sex_availability']], on=['المؤشر', 'الدولة'], how='left')

results.fillna(0).astype(int).to_excel(OUTPUT_FILE, index=False)
print(f"Final report saved to {OUTPUT_FILE}")